In [12]:
import pandas as pd
import networkx as nx
import os
import numpy as np
from collections import deque
from gene_to_uniprot import convert_gene_list#genes--uniprot

DATA_PATH = r"C:\Users\Nisrin Fariss Lamine\Downloads\tfm"

def cargar_datos():
    print("Cargando BioGRID...")
    df_edges = pd.read_csv(os.path.join(DATA_PATH, "biogrid_edges.csv"))
    G = nx.from_pandas_edgelist(df_edges, source="source", target="target")

    print("Cargando DrugBank limpio...")
    df_drug = pd.read_csv(os.path.join(DATA_PATH, "drugbank_targets_clean.csv"))

    print("Cargando Enfermedades...")
    df_disorders = pd.read_csv(os.path.join(DATA_PATH, "disorder_genes.csv"), sep=";")

    return G, df_drug, df_disorders


In [9]:
def bfs_multifuente(grafo, origenes):
    
    distancias = {nodo: float("inf") for nodo in grafo.nodes()}#dist inf,nodos no visitados
    cola = deque()

    for o in origenes:
        if o in distancias:#si esta en nodos no visitados
            distancias[o] = 0# a simismo 
            cola.append(o)

    while cola:
        actual = cola.popleft()
        for vecino in grafo.neighbors(actual):
            if distancias[vecino] == float("inf"):
                distancias[vecino] = distancias[actual] + 1
                cola.append(vecino)

    return distancias#diccionario de nodo y su distancia minima desde  laenfermedad al resto de nodos


In [10]:

G = nx.Graph()
G.add_edges_from([
    ("A", "B"),
    ("B", "C"),
    ("C", "D"),
    ("A", "E")
])
origenes = ["A"]

dist = bfs_multifuente(G, origenes)

print(dist)

{'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 1}


In [ ]:
def construir_bins_por_grado(grafo, tamaño_bin=50):
    
    grados = {nodo: grafo.degree(nodo) for nodo in grafo.nodes()}#conexion de cada nodo
    bins = {}
    for proteina, grado in grados.items():
        bin_id = grado // tamaño_bin #ver a q bin corresponde, eneteros
        bins.setdefault(bin_id, []).append(proteina)
        
    return grados, bins

In [11]:
grados, bins = construir_bins_por_grado(G, tamaño_bin=2)

print("Grados:", grados)
print("Bins:", bins)

Grados: {'A': 2, 'B': 2, 'C': 2, 'D': 1, 'E': 1}
Bins: {1: ['A', 'B', 'C'], 0: ['D', 'E']}
